In [0]:
from pyspark.sql.functions import avg, sum as _sum, col, month, lower

# Rutes
spotify_path = "file:/Workspace/Users/shw5133@gmail.com/Spotify data.csv"
coffee_path = "file:/Workspace/Users/shw5133@gmail.com/coffee_sales.csv"

# files
df_spotify = spark.read.csv(spotify_path, header=True, inferSchema=True)
df_coffee = spark.read.csv(coffee_path, header=True, inferSchema=True)

In [0]:
# Note: Databricks Serverless compute strictly disables RDD APIs 
# Q1 and Q2 were resolved using DataFrames as my ai recomended.

print("--- Q1. Count the number of songs per artist ---")
df_spotify.groupBy("artist").count().withColumnRenamed("count", "total_songs").orderBy(col("total_songs").desc()).show(10)

print("\n--- Q2. Find the average danceability across all songs ---")
df_spotify.select(avg(col("danceability")).alias("avg_danceability")).show()

print("\n--- Q3. Find the total revenue ---")
df_coffee.select(_sum("money").alias("total_revenue")).show()

print("\n--- Q4. Number of transactions for each coffee_name ---")
df_coffee.groupBy("coffee_name").count().withColumnRenamed("count", "total_transactions").show()

print("\n--- Q5. Total revenue per month ---")
df_coffee_month = df_coffee.withColumn("month", month(col("date")))
df_coffee_month.groupBy("month").agg(_sum("money").alias("monthly_revenue")).orderBy("month").show()

--- Q1. Count the number of songs per artist ---
+---------------+-----------+
|         artist|total_songs|
+---------------+-----------+
|          Drake|         16|
|      Rick Ross|         13|
|     Disclosure|         12|
|  WALK THE MOON|         10|
|Backstreet Boys|         10|
|Crystal Castles|          9|
|         FIDLAR|          9|
|         Future|          8|
|    Demi Lovato|          8|
|   Fall Out Boy|          8|
+---------------+-----------+
only showing top 10 rows

--- Q2. Find the average danceability across all songs ---
+------------------+
|  avg_danceability|
+------------------+
|0.6184219137332657|
+------------------+


--- Q3. Find the total revenue ---
+-----------------+
|    total_revenue|
+-----------------+
|115431.5799999978|
+-----------------+


--- Q4. Number of transactions for each coffee_name ---
+-------------------+------------------+
|        coffee_name|total_transactions|
+-------------------+------------------+
|              Cocoa|  

In [0]:
#Q6. Difference between RDD and DataFrame.
#in my undestanding this is just a questions so An RDD is a low-level, distributed collection of objects without a strict schema, offering fine-grained control but lacking native optimization. A DataFrame is a higher-level abstraction built on top of RDDs, organized into named columns with a defined schema, which allows Spark's Catalyst Optimizer to execute queries significantly faster and more efficiently.

print("\n--- Q7. How many times the songs had ‘love’ in them? ---")
love_songs_count = df_spotify.filter(lower(col("song_title")).contains("love")).count()
print(f"Songs with 'love' in the title: {love_songs_count}")

print("\n--- Q8. Curate a party songs list ---")
df_spotify.filter((col("danceability") > 0.8) & (col("energy") > 0.8)).select("song_title", "artist").limit(20).show(truncate=False)


--- Q7. How many times the songs had ‘love’ in them? ---
Songs with 'love' in the title: 88

--- Q8. Curate a party songs list ---
+-----------------------------------------------+----------------------------------+
|song_title                                     |artist                            |
+-----------------------------------------------+----------------------------------+
|I Ain't Trippin off Nothin                     |Ezale                             |
|Sippin On Some Syrup                           |Three 6 Mafia                     |
|Got To Give It Up (Part 1)                     |Marvin Gaye                       |
|WTF (Where They From) [feat. Pharrell Williams]|Missy Elliott                     |
|Talk About                                     |Les Sins                          |
|Bustin' Loose                                  |Chuck Brown and the Soul Searchers|
|More Bounce To The Ounce                       |Zapp                              |
|How Do You See Me

In [0]:
# Convert the DataFrame into a temp SQL
df_coffee.createOrReplaceTempView("coffee_sales")

In [0]:
%sql
-- Q1. Top 5 best-selling coffee types by revenue
SELECT coffee_name, SUM(money) AS total_revenue 
FROM coffee_sales 
GROUP BY coffee_name 
ORDER BY total_revenue DESC 
LIMIT 5;

coffee_name,total_revenue
Latte,27866.299999999457
Americano with Milk,25269.120000000225
Cappuccino,18034.139999999945
Americano,15062.259999999818
Hot Chocolate,10172.460000000045


In [0]:
%sql
-- Q2. Total revenue by payment type (cash_type)
SELECT cash_type, SUM(money) AS total_revenue 
FROM coffee_sales 
GROUP BY cash_type;

cash_type,total_revenue
card,112245.57999999814
cash,3186.0


In [0]:
%sql
-- Q3. Total revenue per month using a CTE
WITH MonthlySales AS (
    SELECT MONTH(date) AS sales_month, money 
    FROM coffee_sales
)
SELECT sales_month, SUM(money) AS total_revenue 
FROM MonthlySales 
GROUP BY sales_month 
ORDER BY sales_month;

sales_month,total_revenue
1,6398.860000000012
2,13215.479999999996
3,17036.639999999945
4,6720.560000000001
5,9063.420000000002
6,7758.760000000005
7,6915.940000000006
8,7613.840000000015
9,9988.640000000007
10,13891.16000000004


In [0]:
%sql
-- Q4. Day with the highest number of transactions
SELECT date, COUNT(*) AS total_transactions 
FROM coffee_sales 
GROUP BY date 
ORDER BY total_transactions DESC 
LIMIT 1;

date,total_transactions
2024-10-11,26


In [0]:
%sql
-- Q5. Rank coffee types by revenue within each month
SELECT 
    coffee_name, 
    MONTH(date) AS sales_month, 
    SUM(money) AS total_revenue,
    RANK() OVER (PARTITION BY MONTH(date) ORDER BY SUM(money) DESC) AS revenue_rank
FROM coffee_sales 
GROUP BY coffee_name, MONTH(date)
ORDER BY sales_month, revenue_rank;

coffee_name,sales_month,total_revenue,revenue_rank
Americano with Milk,1,1604.7199999999984,1
Latte,1,1466.1599999999999,2
Cappuccino,1,965.5199999999999,3
Americano,1,649.0,4
Cortado,1,571.1199999999999,5
Hot Chocolate,1,536.4,6
Cocoa,1,500.63999999999993,7
Espresso,1,105.3,8
Americano,2,3037.3200000000033,1
Americano with Milk,2,2623.0999999999995,2
